In [1]:
import yfinance as yf
import requests 
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
import talib

In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\bmh.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\classic.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test_patch.mplstyle: The text.kerning_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.


In [2]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()  


table = pd.read_html(StringIO(response.text))
df_tickers = table[0]


tickers = df_tickers['Symbol'].str.replace('.', '-', regex=False).tolist()


tickers = tickers[:500]

df3 = yf.download(tickers, start= "2015-05-05", end= "2026-05-05", threads = True)

try:
  close_df = df3["Close"]
  high_df = df3["High"]
  low_df = df3["Low"]
  open_df = df3["Open"]

except KeyError:
  
  close_df = df3.xs("Close", level=0, axis=1)
  high_df = df3.xs("High", level=0, axis=1)
  low_df = df3.xs("Low", level=0, axis=1)
  open_df = df3.xs("Open", level = 0, axis = 1)

close_df = close_df.dropna(axis=1, how="all")
high_df = high_df.dropna(axis=1, how="all")
low_df = low_df.dropna(axis=1, how="all")
open_df = open_df.dropna(axis=1, how = "all")

valid_cols = (
    close_df.columns.intersection(high_df.columns)
    .intersection(low_df.columns).intersection(open_df.columns)
)
close_df, high_df, low_df, open_df = (
    close_df[valid_cols],
    high_df[valid_cols],
    low_df[valid_cols],
    open_df[valid_cols]
)




[**                     5%                       ]  27 of 500 completed$FDXF: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[**********************96%*********************  ]  482 of 500 completed$VYLR: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[**********************99%***********************]  494 of 500 completed$HONA: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[*********************100%***********************]  500 of 500 completed

3 Failed downloads:
['FDXF', 'VYLR', 'HONA']: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")


### Dual Thrust for S and P 500 companies using a long/short strategy for a 5 lookback period with 0.5 sensitivity

In [3]:
HH_df = high_df.rolling(window=5, min_periods=5).max().shift(1)
LC_df = close_df.rolling(window=5, min_periods=5).min().shift(1)

HC_df = close_df.rolling(window=5, min_periods=5).max().shift(1)
LL_df = low_df.rolling(window=5, min_periods=5).min().shift(1)

R1 = HH_df - LC_df
R2 = HC_df - LL_df

R = R1.combine(R2, np.maximum)

upper_limit =  open_df + 0.5 * R 
lower_limit = open_df - 0.5 * R 

upper_hit = high_df >= upper_limit
lower_hit = low_df <= lower_limit

long_signal = upper_hit & ~lower_hit
short_signal = lower_hit & ~upper_hit

long_entry = long_signal.shift(1).fillna(False).astype(bool)
short_entry = short_signal.shift(1).fillna(False).astype(bool)

backtest = vbt.Portfolio.from_signals(
    close = close_df,
    price = open_df,
    entries = long_entry,
    exits = short_entry,
    short_entries= short_entry,
    short_exits=long_entry,
    upon_opposite_entry="reverse",
    init_cash = 100_000.0,
    size=10_000.0,
    size_type="value",
    cash_sharing=True,
    group_by=True,
    freq = "1D",
    fees=0.001

)

print(backtest.stats())
backtest.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                             -8019515.622776
Total Return [%]                         -8119.515623
Benchmark Return [%]                       524.559261
Max Gross Exposure [%]                  134743.142612
Total Fees Paid                         1311901.73828
Max Drawdown [%]                          7331.078668
Max Drawdown Duration              2688 days 00:00:00
Total Trades                                    69152
Total Closed Trades                             68655
Total Open Trades                                 497
Open Trade PnL                        -3370296.091557
Win Rate [%]                                36.488238
Best Trade [%]                              171.20442
Worst Trade [%]                          -1261.676391
Avg Winning Trade [%]       

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



### Controlled position sizings and locking cash

In [4]:

INIT_CASH = 100_000.0
MAX_ALLOCATION = 0.50
N_STOCKS = len(close_df.columns)
ORDER_VALUE = INIT_CASH * MAX_ALLOCATION / N_STOCKS
previous_close = close_df.shift(1)

backtest = vbt.Portfolio.from_signals(
    close=close_df,
    price=open_df,
    val_price=previous_close,
    entries=long_entry,
    exits=short_entry,
    short_entries=short_entry,
    short_exits=long_entry,
    upon_opposite_entry="reverse",
    init_cash=INIT_CASH,
    size=ORDER_VALUE,
    size_type="value",
    cash_sharing=True,
    group_by=True,
    lock_cash=True,
    allow_partial=True,
    fees=0.001,
    slippage=0.0005,
    freq="1D"
)

print(backtest.stats())
backtest.plot().show()


#includes slippage costs and fees for realism



Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                                39096.963756
Total Return [%]                           -60.903036
Benchmark Return [%]                       524.559261
Max Gross Exposure [%]                          100.0
Total Fees Paid                          31975.568605
Max Drawdown [%]                            60.937895
Max Drawdown Duration              2760 days 00:00:00
Total Trades                                   162154
Total Closed Trades                            161657
Total Open Trades                                 497
Open Trade PnL                            1026.456962
Win Rate [%]                                34.482268
Best Trade [%]                             190.565447
Worst Trade [%]                            -83.842656
Avg Winning Trade [%]       

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



### Using trend filters alongside dual thrust

In [5]:
macd_dict = {}
signal_dict = {}
hist_dict = {}
rsi_dict = {}

for ticker in close_df.columns:
    prices = close_df[ticker].to_numpy(dtype=np.float64)

    macd, signal, hist = talib.MACD(
        prices,
        fastperiod=12,
        slowperiod=26,
        signalperiod=9
    )

    rsi = talib.RSI(
        prices,
        timeperiod=14
    )

    macd_dict[ticker] = macd
    signal_dict[ticker] = signal
    hist_dict[ticker] = hist
    rsi_dict[ticker] = rsi

macd_df = pd.DataFrame(macd_dict, index=close_df.index)
signal_df = pd.DataFrame(signal_dict, index=close_df.index)
hist_df = pd.DataFrame(hist_dict, index=close_df.index)
rsi_df = pd.DataFrame(rsi_dict, index = close_df.index )


valid = R.notna() & (R > 0)

upper_hit = valid & (high_df >= upper_limit)
lower_hit = valid & (low_df <= lower_limit)

long_signal = upper_hit & ~lower_hit
short_signal = lower_hit & ~upper_hit


macd_bull = (
    (macd_df >= signal_df)
    .shift(1).fillna(False).astype(bool)
)

macd_bear = (
    (macd_df < signal_df)
    .shift(1).fillna(False).astype(bool)
)

rsi_bull = (
    (rsi_df >= 50)
    .shift(1).fillna(False).astype(bool)
)

rsi_bear = (
    (rsi_df < 50)
    .shift(1).fillna(False).astype(bool)
)

trend_long = macd_bull & rsi_bull
trend_short = macd_bear & rsi_bear


long_entry = long_signal & trend_long
short_entry = short_signal & trend_short

long_exit = short_entry
short_exit = long_entry

execution_price = close_df.copy()

execution_price = execution_price.where(
    ~long_signal, upper_limit
)

execution_price = execution_price.where(
    ~short_signal, lower_limit
)


backtest_1 = vbt.Portfolio.from_signals(
    close=close_df,
    price=execution_price,
    val_price=previous_close,

    entries=long_entry,
    exits=long_exit,
    short_entries=short_entry,
    short_exits=short_exit,

    upon_opposite_entry="reverse",

    init_cash=INIT_CASH,
    size=ORDER_VALUE,
    size_type="value",

    cash_sharing=True,
    group_by=True,
    call_seq="auto",
    lock_cash=True,
    allow_partial=True,

    fees=0.001,
    slippage=0.0005,
    freq="1D"
)

print(backtest_1.stats())
backtest_1.plot().show()



Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                                99408.117494
Total Return [%]                            -0.591883
Benchmark Return [%]                       524.559261
Max Gross Exposure [%]                      59.478916
Total Fees Paid                           6114.433723
Max Drawdown [%]                            12.745736
Max Drawdown Duration              1536 days 00:00:00
Total Trades                                    30223
Total Closed Trades                             29726
Total Open Trades                                 497
Open Trade PnL                            4997.939555
Win Rate [%]                                 35.13759
Best Trade [%]                             435.835766
Worst Trade [%]                            -74.982688
Avg Winning Trade [%]       

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



### Plotting the Equity curve

In [6]:
equity = backtest_1.value(group_by=True)
equity.vbt.plot().show()